<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Clean without hiding demand

**[Clean without hiding demand](https://learning.nextia-ai.com/courses/forecasting/m02/clean-without-hiding-demand/)** · Forecasting and Time Series: Predict What Comes Next · Module 2, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find stock-outs and missing days, fill a missing day, and train a forecast on stock-out days marked as missing instead of as zero demand.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Events as known future inputs, from [Calendars, holidays and events](https://learning.nextia-ai.com/courses/forecasting/m02/calendars-holidays-and-events/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m02/clean-without-hiding-demand/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M02-L04-clean-without-hiding-demand/clean-without-hiding-demand-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A tiny example: sales are not demand

A bakery bakes 30 loaves a day. On day 3 and day 4, it runs out before closing time, so it sells all 30, but more people wanted bread. The bakery counted the customers it sent away.

> **Predict.** The mean of the sales is lower than the mean of the demand. By how much? Then run the cell.

In [ ]:
bakery = pd.DataFrame({"day": [1, 2, 3, 4, 5], "demand": [24, 27, 41, 38, 25]})
bakery["sales"] = bakery["demand"].clip(upper=30)        # no more than the 30 loaves on the shelf
bakery["stock_out"] = (bakery["demand"] > 30).astype(int)
print(bakery.to_string(index=False))
print("mean demand", bakery["demand"].mean(), "| mean sales", bakery["sales"].mean())

> **Check.** You should see `stock_out` 1 on days 3 and 4, and `mean demand 31.0 | mean sales 27.2`. The sales hide 19 loaves of demand. In a real shop, nobody counts the customers who leave: only the flag says that the shelf was empty.

## 2. Find the stock-outs in Larkfield's data

In `demand.csv.gz`, the column `stock_out` is 1 when the shelf was empty for part or all of the day. On those days, the sales are lower than the demand. The real demand is not in the data.

In [ ]:
demand = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
out = demand[demand["stock_out"] == 1]
print(len(out), "stock-out rows of", len(demand), f"({100 * len(out) / len(demand):.1f}%),",
      out.groupby(["product_id", "region"]).ngroups, "product-region series")
p002 = demand[(demand["product_id"] == "P002") & (demand["region"] == "north")].set_index("date")
print(p002.loc["2025-05-08":"2025-05-24", ["sales", "stock_out"]].T.to_string())

> **Check.** You should see `2473 stock-out rows of 403758 (0.6%), 289 product-region series`. P002 (a folding chair) in the north region has `stock_out` 1 from 12 to 20 May 2025, with sales of 0 on all nine days: a situation we planted.

## 3. Train on the zeros, or mark them?

Kenji orders P002 for the north on 20 May, for the next 14 days. The next cell trains statsforecast's `AutoETS` (a method of Module 4; here, read it as "a forecaster") two ways on the history up to 20 May:

- **zeros:** the sales as they are, with the nine stock-out days as 0;
- **marked:** the stock-out days marked as missing, then filled with the mean of the same weekday one and two weeks before.

> **Predict.** Which forecast for the next 14 days will be closer to what P002 really sold? Then run the cell.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

history = p002.loc[:"2025-05-20", ["sales", "stock_out"]].reset_index()
zeros = history["sales"].astype(float)
marked = zeros.where(history["stock_out"] == 0)                 # stock-out days become missing (NaN)
marked = marked.fillna((marked.shift(7) + marked.shift(14)) / 2).fillna(marked.shift(14)).ffill()
real = int(p002.loc["2025-05-21":"2025-06-03", "sales"].sum())

for name, y in [("zeros", zeros), ("marked", marked)]:
    sf = StatsForecast(models=[AutoETS(season_length=7)], freq="D")
    fc = sf.forecast(df=pd.DataFrame({"unique_id": "P002/north", "ds": history["date"], "y": y}), h=14)
    total = fc["AutoETS"].sum()
    print(f"{name:7} 14-day forecast {total:5.1f}, real sales {real}, bias {100 * (total - real) / real:+.1f}%")
print("filled stock-out days:", marked[history["stock_out"] == 1].tolist())

> **Check.** You should see `zeros   14-day forecast   7.5, real sales 52, bias -85.6%` and `marked  14-day forecast  27.9, real sales 52, bias -46.3%`. Trained on the zeros, the forecast thinks the chair stopped selling. Marking the days does not make the forecast right: it is still too low, because the filled values come from April, before the season started, and because a promotion week (26 May to 1 June) falls inside the 14 days. But it is four times closer.

## 4. The missing day

On 19 October 2025, the sales and orders feed dropped a day. There are no rows at all for that date. The next cell shows the orders around it, then fills the gap with the mean of the same weekday one week before and one week after.

In [ ]:
raw = pd.read_csv("data/orders.csv", parse_dates=["date"]).groupby("date")["orders"].sum()
print("rows on 2025-10-19:", int((raw.index == "2025-10-19").sum()))
daily = raw.asfreq("D")                                      # one row per day: the missing day becomes NaN
print(daily["2025-10-17":"2025-10-21"].to_string())
filled = daily.fillna((daily.shift(7) + daily.shift(-7)) / 2).round()
print("one week before:", int(daily["2025-10-12"]), "| one week after:", int(daily["2025-10-26"]),
      "| filled:", int(filled["2025-10-19"]))

> **Check.** You should see `rows on 2025-10-19: 0`, a `NaN` on 2025-10-19 after `asfreq("D")`, and `one week before: 695 | one week after: 592 | filled: 644` (the mean, 643.5, rounded).

Without `asfreq("D")`, the gap is invisible: the series just jumps from 18 to 20 October, and every "7 days before" after it points at the wrong weekday.

## 5. A real spike: keep it

The heat-wave week of 30 June to 6 July 2025 is a situation we planted. Parasol sales jumped, and that was real demand. The next cell shows the parasol's weekly sales around it.

In [ ]:
sales = load_sales(["P025"]).set_index("ds")["y"]
weekly = sales["2025-06-16":"2025-07-20"].resample("W-SUN").sum()
print(weekly.astype(int).to_string())

> **Check.** You should see `2025-07-06    147`, against `50` in the week before. The spike is real demand, not an error. Do not delete it: mark it, keep it, and check how much weight the next forecast gives it. Finding unusual points like this one in many series is the subject of the anomaly detection course.

## 6. Your turn: which product is the hardest?

> **Your turn.** For three products, P025 (parasol), P077 (garden hose) and P141 (duvet), build a table `summary` with one row per product (the index is the product) and three columns, in this order:
>
> - `stock_out_days`: the days with `stock_out` 1 in `load_sales()` (any region out of stock);
> - `promotion_days`: the days with `on_promotion` 1;
> - `remainder_share`: the remainder's share of the variance in %, rounded to 1 decimal, from `MSTL(season_length=[7, 365])` as in the first lesson of this module.
>
> Then run the check cell, and decide which product will be the hardest to forecast, and why.

In [ ]:
summary = ...  # a DataFrame: index = product, columns stock_out_days, promotion_days, remainder_share

In [ ]:
expect_hash("summary", None if summary is ... else summary.reset_index(drop=True), "94324dd5571d3e4b")

> **Check.** The check prints `Check passed`. P025 has 9 stock-out days, 14 promotion days and a remainder share of 19.1%; P077 has 0, 14 and 27.7%; P141 has 13, 7 and 32.7%.

P141 has the largest remainder share and the most stock-out days: its promotions are not in the seasons, so they stay in the remainder, and its stock-outs hide demand. With a promotion column and marked stock-outs, a forecast can explain much of it. P077 has no stock-out, but its price rise in June 2025 moves its level for good, and a decomposition puts that change in the trend and the remainder. The parasol has the strongest season, and its stock-out days in the heat wave hide some demand. Each product is hard in its own way, and the parts tell you why.

## Next

You can now clean a series without hiding demand. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m02/clean-without-hiding-demand/) has the charts and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Naive forecasts](https://learning.nextia-ai.com/courses/forecasting/m03/naive-forecasts/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m02/clean-without-hiding-demand/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.